In [ ]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

RAW_CSV_PATH = os.path.join(DATA_RAW_DIR, "WELFake_Dataset.csv")

df = pd.read_csv(RAW_CSV_PATH)

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["full_text"] = (df["title"].str.strip() + ". " + df["text"].str.strip()).str.strip()

df = df[df["full_text"].str.len() > 20].copy()

df["label"] = df["label"].astype(int)

df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, "val.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

train_df[["full_text", "label"]].to_csv(train_path, index=False)
val_df[["full_text", "label"]].to_csv(val_path, index=False)
test_df[["full_text", "label"]].to_csv(test_path, index=False)

print("Saved:")
print(" -", train_path, train_df.shape)
print(" -", val_path, val_df.shape)
print(" -", test_path, test_df.shape)

print("\nLabel distribution (train):")
print(train_df["label"].value_counts(normalize=True).round(4))

print("\nLabel distribution (val):")
print(val_df["label"].value_counts(normalize=True).round(4))

print("\nLabel distribution (test):")
print(test_df["label"].value_counts(normalize=True).round(4))

lengths = df["full_text"].str.split().apply(len)
print("\nFull_text length (tokens approx) stats:")
print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))